# Assign interfaces prototyping

Can I assign interface labels to connections in an ENM?

Pre-exercises:

1. Create a table of links connecting with atoms in the ASU
2. Identify the unique interfaces, without taking symmetry into account
3. Create a mapping between the symops and their inverses, modulo unit cell translations
4. Assign labels to the interfaces

## Ex. 3

Create a mapping between the symops and their inverses, modulo unit cell translations.

I'm starting here, because it's central to the whole scheme and I don't want to get bogged down with a big table just yet.

In [6]:
import gemmi
import pandas as pd

st = gemmi.read_structure('test_data/lys_1_refmac.pdb')
st.setup_entities() # supposed to be good practice
st[0].remove_ligands_and_waters()

In [3]:
# structure contains a list of transforms generating all the images in the unit cell, modulo PBC shift
st.cell.images[0]

In [4]:
def is_identity(t,wrap=True,tol=1e-9):
    """return True if the transform is identity modulo PBC"""
    tf = False
    if t.mat.approx(gemmi.Mat33(),tol):
        fvec = gemmi.Fractional(*t.vec.tolist())
        if wrap:
            fvec = fvec.wrap_to_zero()
        tf = fvec.approx(gemmi.Vec3(0,0,0),tol)
    return tf

def find_inverse(t, images, wrap=True):
    """Find the inverse of a transform (t) from within a list of transforms (images). Returns the first matching index"""
    for j, im in enumerate(images):
        if is_identity(t.combine(im), wrap=wrap):
            return j

image_pairs = [(j, find_inverse(t, st.cell.images, wrap=True)) for j, t in enumerate(st.cell.images)]
image_pairs

[(0, 2), (1, 1), (2, 0), (3, 3), (4, 4), (5, 5), (6, 6)]

### Ex. 1

Create a table of links connecting with atoms in the ASU

In [ ]:

# from proto-create-enm.ipynb

cs = gemmi.ContactSearch(4.0)
cs.ignore = gemmi.ContactSearch.Ignore.SameAsu # WARNING! will break for P1, and other low symmetry space groups
ns = gemmi.NeighborSearch(st[0], st.cell, 5).populate(include_h=False)
results = cs.find_contacts(ns)

def results2dict(contacts):
    d = {
        'cra1':[str(res.partner1) for res in contacts],
        'cra2':[str(res.partner2) for res in contacts],
        'sym_idx1':[0 for res in contacts],
        'sym_idx2':[res.image_idx for res in contacts],
        'pbc_shift1':[(0,0,0) for res in contacts],
        'pbc_shift2':[st.cell.find_nearest_pbc_image(
            res.partner1.atom.pos, 
            res.partner2.atom.pos, 
            res.image_idx).pbc_shift for res in contacts],
    }
    return d

df = pd.DataFrame.from_dict(results2dict(results))
df

,cra1,cra2,sym_idx1,sym_idx2,pbc_shift1,pbc_shift2
0,A/ALA 10/CB,A/ARG 14/CD,0,5,"(0, 0, 0)","(0, 0, 0)"
1,A/LYS 13/CE,A/LEU 129/O,0,5,"(0, 0, 0)","(0, 0, 0)"
2,A/LYS 13/CE,A/LEU 129/C,0,5,"(0, 0, 0)","(0, 0, 0)"
3,A/LYS 13/NZ,A/LEU 129/O,0,5,"(0, 0, 0)","(0, 0, 0)"
4,A/LYS 13/NZ,A/LEU 129/C,0,5,"(0, 0, 0)","(0, 0, 0)"
...,...,...,...,...,...,...
97,A/ASN 106/ND2,A/ASN 113/CG,0,3,"(0, 0, 0)","(-1, 0, 0)"
98,A/ASN 106/ND2,A/ASN 113/OD1,0,3,"(0, 0, 0)","(-1, 0, 0)"
99,A/ASN 113/O,A/LYS 116/CE,0,1,"(0, 0, 0)","(0, 0, -1)"
100,A/ASN 113/O,A/LYS 116/NZ,0,1,"(0, 0, 0)","(0, 0, -1)"


### Ex. 2

Identify the unique interfaces, without taking symmetry into account

In [8]:
df.groupby(['sym_idx1','pbc_shift1','sym_idx2','pbc_shift2']).size() #.reset_index(name='count')

sym_idx1  pbc_shift1  sym_idx2  pbc_shift2
0         (0, 0, 0)   1         (0, 0, -1)     2
                                (0, 0, 0)     15
                      3         (-1, 0, 0)    37
                      5         (0, 0, 0)     13
                      7         (0, 0, 1)     33
                                (0, 0, 2)      2
dtype: int64

There are 6 unique interfaces, at least as we've defined it.

In [9]:
df['interface'] = df.groupby(['sym_idx1','pbc_shift1','sym_idx2','pbc_shift2']).ngroup()
df

,cra1,cra2,sym_idx1,sym_idx2,pbc_shift1,pbc_shift2,interface
0,A/ALA 10/CB,A/ARG 14/CD,0,5,"(0, 0, 0)","(0, 0, 0)",3
1,A/LYS 13/CE,A/LEU 129/O,0,5,"(0, 0, 0)","(0, 0, 0)",3
2,A/LYS 13/CE,A/LEU 129/C,0,5,"(0, 0, 0)","(0, 0, 0)",3
3,A/LYS 13/NZ,A/LEU 129/O,0,5,"(0, 0, 0)","(0, 0, 0)",3
4,A/LYS 13/NZ,A/LEU 129/C,0,5,"(0, 0, 0)","(0, 0, 0)",3
...,...,...,...,...,...,...,...
97,A/ASN 106/ND2,A/ASN 113/CG,0,3,"(0, 0, 0)","(-1, 0, 0)",2
98,A/ASN 106/ND2,A/ASN 113/OD1,0,3,"(0, 0, 0)","(-1, 0, 0)",2
99,A/ASN 113/O,A/LYS 116/CE,0,1,"(0, 0, 0)","(0, 0, -1)",0
100,A/ASN 113/O,A/LYS 116/NZ,0,1,"(0, 0, 0)","(0, 0, -1)",0


### Ex. 4

Assign labels to the interfaces

In [10]:
row = df.loc[12] # choose this one for now
row

cra1             A/GLY 22/C
cra2          A/ARG 114/NH1
sym_idx1                  0
sym_idx2                  3
pbc_shift1        (0, 0, 0)
pbc_shift2       (-1, 0, 0)
interface                 2
Name: 12, dtype: object

In [11]:
sym_pair_lookup = {(j+1):(k+1) for j, k in sorted(image_pairs)}
sym_pair_lookup[0] = 0
# j = row['sym_idx2']
# k = sym_pair_lookup[j]
# if k < j:
#     print(f"{j} --> -{k}")

sym_labels = {}

for j, k in sorted(image_pairs):
    if k < j: # label j using -k instead
        sym_labels[j+1] = -(k+1)
    else: # label j with itself
        sym_labels[j+1] = (j+1)

pd.Series(sym_labels).sort_index()

1    1
2    2
3   -1
4    4
5    5
6    6
7    7
dtype: int64

### Confusion

WAIT... this kinda makes sense, except that I don't know what to do about interfaces with different PBC shifts and the same symop. how did I handle it in MATLAB?

In MATLAB, I this lookup:

```octave
[ia,locb] = ismember(arrayfun(@inv,o),o);
assert(all(ia));
```

That's interesting! I really do expect all of the inverses to be present. Lets make a list of ASU neighbor ops.

In [12]:
df2 = df.drop_duplicates(subset=['interface'], keep='first').set_index('interface')[['sym_idx2','pbc_shift2']].sort_index()
df2

,sym_idx2,pbc_shift2
interface,,
0,1,"(0, 0, -1)"
1,1,"(0, 0, 0)"
2,3,"(-1, 0, 0)"
3,5,"(0, 0, 0)"
4,7,"(0, 0, 1)"
5,7,"(0, 0, 2)"


In [13]:
# create neighbor ops for each row

neighbor_ops = []

for index, row in df2.iterrows():
    #print(index, row['sym_idx2'], row['pbc_shift2'])
    if row['sym_idx2'] == 0:
        neighbor_op = gemmi.Transform()
    else:
        image_transform = st.cell.images[row['sym_idx2'] - 1]
        pbc_transform = gemmi.Transform(gemmi.Mat33(), gemmi.Vec3(*row['pbc_shift2']))
        neighbor_op = pbc_transform.combine(image_transform)
    neighbor_ops.append(neighbor_op)
    
neighbor_ops

In [14]:
neighbor_pairs = [(j, find_inverse(t, neighbor_ops, wrap=False)) for j, t in enumerate(neighbor_ops)]
neighbor_pairs

[(0, 2), (1, None), (2, 0), (3, 3), (4, 4), (5, 5)]

In [15]:
n1inv = neighbor_ops[1].inverse()
n1inv.mat, n1inv.vec

(<gemmi.Mat33 [0, 1, 0]
              [-1, 0, 0]
              [0, 0, 1]>,
 <gemmi.Vec3(-0.5, 0.5, -0.75)>)

In [16]:
for o in neighbor_ops:
    if o.mat.approx(n1inv.mat,1e-2):
        print(o.vec)

<gemmi.Vec3(-0.5, 0.5, 0.25)>


Does MATLAB `__eq__` function on operators wrap?

No. So why is this not showing up in MATLAB?

In [17]:
# is the inverse being computed correctly?
im = st.cell.images[0]
n1inv.mat.approx(im.mat.inverse(),1e-9), n1inv.vec.approx(-1*im.mat.inverse().multiply(im.vec),1e-9)

(True, True)

In [18]:
# this makes me wonder if the PBC op is being recovered correctly
# to test, lets go through the table and compute distances manually

def address_from_cra_string(cra):
    chain, residue, atom = cra.split('/')
    resname, seqid = residue.split()
    if '.' in atom:
        atom, altloc = atom.split('.')
    else:
        altloc = '\x00'
    addr =  gemmi.AtomAddress(chain, gemmi.SeqId(seqid), resname, atom, altloc)
    return addr

def get_image_transform(sym_idx, pbc_shift):
    if sym_idx == 0:
        return gemmi.Transform()
    image_transform = st.cell.images[sym_idx - 1]
    pbc_transform = gemmi.Transform(gemmi.Mat33(), gemmi.Vec3(*pbc_shift))
    return pbc_transform.combine(image_transform)

def apply_transform_to_position(t,p):
    return st.cell.orthogonalize(gemmi.Fractional(t.apply(st.cell.fractionalize(p))))
    
def calculate_distance(row):
    cra1 = st[0].find_cra(address_from_cra_string(row['cra1']))
    cra2 = st[0].find_cra(address_from_cra_string(row['cra2']))
    t = get_image_transform(row['sym_idx2'],row['pbc_shift2'])
    pos2 = apply_transform_to_position(t,cra2.atom.pos)
    return (cra1.atom.pos - pos2).length()

df.apply(calculate_distance, axis=1).max()

np.float64(3.9930383168710004)

In [19]:
# nope! symmetry operators are totally correct. It's my thinking, apparently, that is not correct.

# what about the inverse distance?
def calculate_inverse_distance(row):
    cra1 = st[0].find_cra(address_from_cra_string(row['cra1']))
    cra2 = st[0].find_cra(address_from_cra_string(row['cra2']))
    t = get_image_transform(row['sym_idx2'],row['pbc_shift2'])
    pos1 = apply_transform_to_position(t.inverse(),cra1.atom.pos)
    return (cra2.atom.pos - pos1).length()

df.apply(calculate_distance, axis=1).max()
# shit, this is also correct.


np.float64(3.9930383168710004)

In [20]:
# are all the cra2 atoms also present in the list of cra1?

df['cra2'].isin(df['cra1']).all()

np.False_

In [21]:
# OH, why is that?
#
# maybe I need to use "twice=True" in contactsearch?

df[~df['cra2'].isin(df['cra1'])]

,cra1,cra2,sym_idx1,sym_idx2,pbc_shift1,pbc_shift2,interface
0,A/ALA 10/CB,A/ARG 14/CD,0,5,"(0, 0, 0)","(0, 0, 0)",3
2,A/LYS 13/CE,A/LEU 129/C,0,5,"(0, 0, 0)","(0, 0, 0)",3
4,A/LYS 13/NZ,A/LEU 129/C,0,5,"(0, 0, 0)","(0, 0, 0)",3
5,A/LYS 13/O,A/ARG 128/NE,0,5,"(0, 0, 0)","(0, 0, 0)",3
6,A/ARG 14/O,A/ARG 128/NH2,0,5,"(0, 0, 0)","(0, 0, 0)",3
...,...,...,...,...,...,...,...
95,A/ASN 106/ND2,A/ASN 113/C,0,3,"(0, 0, 0)","(-1, 0, 0)",2
97,A/ASN 106/ND2,A/ASN 113/CG,0,3,"(0, 0, 0)","(-1, 0, 0)",2
98,A/ASN 106/ND2,A/ASN 113/OD1,0,3,"(0, 0, 0)","(-1, 0, 0)",2
99,A/ASN 113/O,A/LYS 116/CE,0,1,"(0, 0, 0)","(0, 0, -1)",0


In [22]:
cs = gemmi.ContactSearch(4.0)
cs.ignore = gemmi.ContactSearch.Ignore.SameAsu # WARNING! will break for P1, and other low symmetry space groups
cs.twice=True # get both copies?
ns = gemmi.NeighborSearch(st[0], st.cell, 5).populate(include_h=False)
results = cs.find_contacts(ns)

def results2dict(contacts):
    d = {
        'cra1':[str(res.partner1) for res in contacts],
        'cra2':[str(res.partner2) for res in contacts],
        'sym_idx1':[0 for res in contacts],
        'sym_idx2':[res.image_idx for res in contacts],
        'pbc_shift1':[(0,0,0) for res in contacts],
        'pbc_shift2':[st.cell.find_nearest_pbc_image(
            res.partner1.atom.pos, 
            res.partner2.atom.pos, 
            res.image_idx).pbc_shift for res in contacts],
    }
    return d

df4 = pd.DataFrame.from_dict(results2dict(results))
df4

,cra1,cra2,sym_idx1,sym_idx2,pbc_shift1,pbc_shift2
0,A/ALA 10/CB,A/ARG 14/CD,0,5,"(0, 0, 0)","(0, 0, 0)"
1,A/LYS 13/CE,A/LEU 129/O,0,5,"(0, 0, 0)","(0, 0, 0)"
2,A/LYS 13/CE,A/LEU 129/C,0,5,"(0, 0, 0)","(0, 0, 0)"
3,A/LYS 13/NZ,A/LEU 129/O,0,5,"(0, 0, 0)","(0, 0, 0)"
4,A/LYS 13/NZ,A/LEU 129/C,0,5,"(0, 0, 0)","(0, 0, 0)"
...,...,...,...,...,...,...
197,A/LEU 129/C,A/LYS 13/CE,0,5,"(0, 0, 0)","(0, 0, 0)"
198,A/LEU 129/C,A/LYS 13/NZ,0,5,"(0, 0, 0)","(0, 0, 0)"
199,A/LEU 129/O,A/LYS 13/CE,0,5,"(0, 0, 0)","(0, 0, 0)"
200,A/LEU 129/O,A/LYS 13/NZ,0,5,"(0, 0, 0)","(0, 0, 0)"


In [23]:
df4['cra2'].isin(df4['cra1']).all()

np.True_

In [24]:
df4['interface'] = df4.groupby(['sym_idx1','pbc_shift1','sym_idx2','pbc_shift2']).ngroup()
df5 = df4.drop_duplicates(subset=['interface'], keep='first').set_index('interface')[['sym_idx2','pbc_shift2']].sort_index()
df5

,sym_idx2,pbc_shift2
interface,,
0,1,"(0, 0, -1)"
1,1,"(0, 0, 0)"
2,3,"(-1, 0, -1)"
3,3,"(-1, 0, 0)"
4,5,"(0, 0, 0)"
5,7,"(0, 0, 1)"
6,7,"(0, 0, 2)"


In [25]:
neighbor_ops = []

for index, row in df5.iterrows():
    #print(index, row['sym_idx2'], row['pbc_shift2'])
    if row['sym_idx2'] == 0:
        neighbor_op = gemmi.Transform()
    else:
        image_transform = st.cell.images[row['sym_idx2'] - 1]
        pbc_transform = gemmi.Transform(gemmi.Mat33(), gemmi.Vec3(*row['pbc_shift2']))
        neighbor_op = pbc_transform.combine(image_transform)
    neighbor_ops.append(neighbor_op)
    
neighbor_ops

In [26]:
neighbor_pairs = [(j, find_inverse(t, neighbor_ops, wrap=False)) for j, t in enumerate(neighbor_ops)]
neighbor_pairs

[(0, 3), (1, 2), (2, 1), (3, 0), (4, 4), (5, 5), (6, 6)]

**AH HA! we need both**